# Compute DistilBERT Embeddings and Linguistic Features (Colab)

This notebook computes CLS embeddings using `distilbert-base-multilingual-cased` and extracts the 7 linguistic features from the repo code. It saves `embeddings.npy`, `features.npy`, and a trained `hybrid_model_colab.pkl`.

Instructions: upload this notebook to Colab (File → Upload notebook) or open it from Drive, then run cells in order. If you prefer, mount Google Drive to save artifacts there.

In [ ]:
# Install dependencies (runs once)
!pip install -q transformers torch scikit-learn datasets pandas numpy fastapi uvicorn joblib imbalanced-learn

In [ ]:
# Optional: mount Google Drive to save artifacts persistently
from google.colab import drive
# Uncomment the next line if you want to mount Drive
# drive.mount('/content/drive')

# Optionally set an output folder inside your Drive after mounting, e.g.:
# OUT_DIR = '/content/drive/MyDrive/taglish_artifacts'
# import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Helper: move uploaded /content/corpus.csv into the repo layout so the notebook can run without cloning
import os, shutil
REPO_DIR = '/content/taglish-detector'
os.makedirs(os.path.join(REPO_DIR,'data','raw'), exist_ok=True)
src = '/content/corpus.csv'  # upload via Files pane

dst = os.path.join(REPO_DIR,'data','raw','corpus.csv')
if os.path.exists(src):
    shutil.move(src, dst)
    print('Moved corpus.csv ->', dst)
else:
    print('No /content/corpus.csv found. Upload it from the left Files pane. Current /content files:', os.listdir('/content')[:50])
print('Target exists:', os.path.exists(dst))

In [ ]:
# Provide the path to the repository root (where `taglish-detector` lives).
# If you cloned your repo into the Colab VM, set REPO_DIR accordingly.
# If you upload only the `taglish-detector` folder, set REPO_DIR = '/content/taglish-detector'
REPO_DIR = '/content/taglish-detector'
import os
if not os.path.exists(REPO_DIR):
  print('Note: REPO_DIR does not exist. If you have the repo online, you can git clone it:')
  print('  !git clone <your-repo-url> repo && mv repo/taglish-detector /content/taglish-detector')
else:
  print('REPO_DIR found:', REPO_DIR)

# Add repo to python path so imports from src.* work
import sys
if REPO_DIR not in sys.path:
  sys.path.insert(0, REPO_DIR)

# Verify expected files exist
print('Checking key files:')
print('-', os.path.exists(os.path.join(REPO_DIR,'data','raw','corpus.csv')), 'data/raw/corpus.csv')
print('-', os.path.exists(os.path.join(REPO_DIR,'src','embedding_extractor.py')), 'src/embedding_extractor.py')
print('-', os.path.exists(os.path.join(REPO_DIR,'src','feature_extractor.py')), 'src/feature_extractor.py')

In [ ]:
# Preprocess and save cleaned CSV using the repo's code.
# This mirrors what we ran locally: Taglish filter + cleaning.
import os, pandas as pd
from src.taglish_filter import filter_taglish
from src.preprocessing import preprocess_df
RAW = os.path.join(REPO_DIR, 'data', 'raw', 'corpus.csv')
OUT = os.path.join(REPO_DIR, 'data', 'processed', 'clean.csv')
print('RAW path =', RAW)
df = pd.read_csv(RAW)
# rename if needed (HuggingFace dataset column name => article)
if 'article' in df.columns and 'text' not in df.columns:
  df = df.rename(columns={'article':'text'})
print('Raw rows:', len(df))
f = filter_taglish(df)
print('After Taglish filter rows:', len(f))
clean = preprocess_df(f, text_col='text')
os.makedirs(os.path.dirname(OUT), exist_ok=True)
clean.to_csv(OUT, index=False)
print('Saved cleaned file with', len(clean), 'rows to', OUT)

In [ ]:
# Compute embeddings (CLS) and the 7 linguistic features, then save as .npy
import os, numpy as np, pandas as pd
from src.embedding_extractor import embeddings_for
from src.feature_extractor import batch_extract
CLEAN = os.path.join(REPO_DIR, 'data', 'processed', 'clean.csv')
df = pd.read_csv(CLEAN)
texts = df['clean_text'].astype(str).tolist()
print('Samples to process:', len(texts))
# Compute embeddings (this will download model on first run)
embeds = embeddings_for(texts)
print('Embeddings shape:', embeds.shape)
feats = batch_extract(texts)
print('Features shape:', feats.shape)
np.save(os.path.join(REPO_DIR, 'data', 'processed', 'embeddings.npy'), embeds)
np.save(os.path.join(REPO_DIR, 'data', 'processed', 'features.npy'), feats)
print('Saved embeddings.npy and features.npy to data/processed')

In [ ]:
# Quick training of a LogisticRegression hybrid model (optional)
import os, joblib, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
EMB_PATH = os.path.join(REPO_DIR, 'data', 'processed', 'embeddings.npy')
FEAT_PATH = os.path.join(REPO_DIR, 'data', 'processed', 'features.npy')
clean_path = os.path.join(REPO_DIR, 'data', 'processed', 'clean.csv')
X_emb = np.load(EMB_PATH)
X_feat = np.load(FEAT_PATH)
X = np.hstack([X_emb, X_feat])
y = pd.read_csv(clean_path)['label'].values
print('X shape, y shape:', X.shape, y.shape)
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
pipe.fit(X, y)
MODEL_PATH = os.path.join(REPO_DIR, 'models', 'hybrid_model_colab.pkl')
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(pipe, MODEL_PATH)
print('Saved model to', MODEL_PATH)

In [ ]:
# Optionally copy artifacts to Drive (uncomment if Drive is mounted)
# import shutil
# shutil.copy(os.path.join(REPO_DIR,'data','processed','embeddings.npy'), OUT_DIR)
# shutil.copy(os.path.join(REPO_DIR,'data','processed','features.npy'), OUT_DIR)
# shutil.copy(MODEL_PATH, OUT_DIR)
print('If Drive mounted, copy files to OUT_DIR as needed.')

**Next steps**:
- Download `embeddings.npy` and `features.npy` (or copy from Drive) and run training locally to run multiple experiments quickly.
- Or train full experiment in Colab and save `hybrid_model_colab.pkl` to Drive.
- After you have a saved model, move it to `taglish-detector/models/hybrid_model.pkl` and start the API: `uvicorn src.api:app --reload`.